# Multi-agent DDPG for MPE (Multi-Agent Particle Environment)


* MADDPG: Multi-Agent Deep Deterministic Policy Gradient (Lowe et al., 2017) [paper](https://arxiv.org/abs/1706.02275)
* MPE: Multi-Agent Particle Environment using pettingzoo library [github](https://github.com/Farama-Foundation/PettingZoo)
* DDPG: Deep Deterministic Policy Gradient (Lillicrap et al., 2015) [paper](https://arxiv.org/abs/1509.02971)

## MADDPG Algorithm

**Multi-agent DDPG (MADDPG)** (Lowe et al., 2017) extends DDPG to an environment where multiple agents are coordinating to complete tasks with only local information. In the viewpoint of one agent, the environment is non-stationary as policies of other agents are quickly upgraded and remain unknown. MADDPG is an actor-critic model redesigned particularly for handling such a changing environment and interactions between agents.

The problem can be formalized in the multi-agent version of MDP, also known as _Markov games_. MADDPG is proposed for partially observable Markov games. Say, there are $N$ agents in total with a set of states $\mathcal{S}$. Each agent owns a set of possible actions, $\mathcal{A}_1, \dots, \mathcal{A}_N$, and a set of observation, $\mathcal{O}_1, \dots, \mathcal{O}_N$. The state transition function involves all states, action and observation spaces $\mathcal{T}: \mathcal{S} \times \mathcal{A}_1 \times ... \times \mathcal{A}_N \rightarrow \mathcal{S}$. Each agent's stochastic policy only involves its own state and action: $\pi_{\theta_i}: \mathcal{O}_i \times \mathcal{A}_i \mapsto [0, 1]$, a probability distribution over actions given its own observation, or a deterministic policy: $\mu_{\theta_i}: \mathcal{O}_i \mapsto \mathcal{A}_i$.

Let $\vec{o} = {o_1, \dots, o_N}, \vec{\mu} = {\mu_1, \dots, \mu_N}$ and the policies are parameterized by $\vec{\theta} = {\theta_1, \dots, \theta_N}$.

The critic in MADDPG learns a centralized action-value function $Q^\mu_i(\vec{o}, a_1, \dots, a_N)$ for the i-th agent, where $a_1 \in \mathcal{A}_1, \dots, a_N \in \mathcal{A}_N$ are actions of all agents. Each $Q^\mu_i$ is learned separately for $i=1, \dots, N$ and therefore multiple agents can have arbitrary reward structures, including conflicting rewards in a competitive setting. Meanwhile, multiple actors, one for each agent, are exploring and upgrading the policy parameters $\theta_i$ on their own.

### Actor update:

$$
\nabla_{\theta_i} J(\theta_i) = \mathbb{E}_{\vec{o}, a \sim \mathcal{D}} [\nabla_{a_i} Q^{\vec{\mu}}_i (\vec{o}, a_1, \dots, a_N) \nabla_{\theta_i} \mu_{\theta_i}(o_i) \rvert_{a_i=\mu_{\theta_i}(o_i)} ]
$$

Where $\mathcal{D}$ is the memory buffer for experience replay, containing multiple episode samples $(\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}’)$ — given current observation $\vec{o}$, agents take action $a_1, \dots, a_N$ and get rewards $r_1, \dots, r_N$, leading to the new observation $\vec{o}’$.

### Critic update:

$$
\begin{aligned}
\mathcal{L}(\theta_i) &= \mathbb{E}_{\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}'}[ (Q^{\vec{\mu}}_i(\vec{o}, a_1, \dots, a_N) - y)^2 ] & \\
\text{where } y &= r_i + \gamma Q^{\vec{\mu}'}_i (\vec{o}', a'_1, \dots, a'_N) \rvert_{a'_j = \mu'_{\theta_j}((o'_j))} & \scriptstyle{\text{; TD target!}}
\end{aligned}
$$
 
where $\vec{\mu}’$ are the target policies with delayed softly-updated parameters.

If the policies $\vec{\mu}$ are unknown during the critic update, we can ask each agent to learn and evolve its own approximation of others' policies. Using the approximated policies, MADDPG still can learn efficiently although the inferred policies might not be accurate.

To mitigate the high variance triggered by the interaction between competing or collaborating agents in the environment, MADDPG proposed one more element - _policy ensembles_:

1. Train K policies for one agent;
2. Pick a random policy for episode rollouts;
3. Take an ensemble of these K policies to do gradient update.

In summary, MADDPG added three additional ingredients on top of DDPG to make it adapt to the multi-agent environment:

* Centralized critic + decentralized actors;
* Actors are able to use estimated policies of other agents for learning;
* Policy ensembling is good for reducing variance.

<div style="text-align:center"><img src="../../assets/images/MADDPG.png" width="600" height="auto"></div>


Here is the final algorithm:

<div style="text-align:center"><img src="../../assets/images/MADDPG-algorithm.png" width="750" height="auto"></div>

### ENVIRONMENT

We going to use PettingZoo library to create the environment. And our environment is MPE (Multi-Agent Particle Environment). The environment is a simple grid-world with agents and landmarks.



In [23]:
from pettingzoo.mpe import simple_spread_v3
import numpy as np

env = simple_spread_v3.parallel_env(continuous_actions=True)
    
# Reset the environment
observations, infos = env.reset()

# Print basic environment information
print(f"Environment: {env.metadata['name']}")
print(f"Number of agents: {len(env.agents)}")
print("Agents:")
for agent in env.agents:
    print(f"  - {agent}")
print("\nAction Spaces:")
for agent in env.agents:
    action_space = env.action_space(agent)
    print(f"  - {agent}: {action_space} (shape: {action_space.shape}, bounds: [{action_space.low}, {action_space.high}])")
print("\nObservation Spaces:")
for agent in env.agents:
    obs_space = env.observation_space(agent)
    print(f"  - {agent}: {obs_space} (shape: {obs_space.shape})")
print("\nSample Observation Shapes:")
for agent in observations:
    print(f"  - {agent}: {np.array(observations[agent]).shape}")
actions = {agent: env.action_space(agent).sample() for agent in env.agents}
next_obs, rewards, terminations, truncations, infos = env.step(actions)
print("\nRewards received:")
print(rewards)
for agent, reward in rewards.items():
    print(f"  - {agent}: {reward}")

print("\nTerminations:")
for agent, terminated in terminations.items():
    print(f"  - {agent}: {terminated}")

print("\nNext Observations:")
for agent, obs in next_obs.items():
    print(f"  - {agent}: {np.array(obs)}")

Environment: simple_spread_v3
Number of agents: 3
Agents:
  - agent_0
  - agent_1
  - agent_2

Action Spaces:
  - agent_0: Box(0.0, 1.0, (5,), float32) (shape: (5,), bounds: [[0. 0. 0. 0. 0.], [1. 1. 1. 1. 1.]])
  - agent_1: Box(0.0, 1.0, (5,), float32) (shape: (5,), bounds: [[0. 0. 0. 0. 0.], [1. 1. 1. 1. 1.]])
  - agent_2: Box(0.0, 1.0, (5,), float32) (shape: (5,), bounds: [[0. 0. 0. 0. 0.], [1. 1. 1. 1. 1.]])

Observation Spaces:
  - agent_0: Box(-inf, inf, (18,), float32) (shape: (18,))
  - agent_1: Box(-inf, inf, (18,), float32) (shape: (18,))
  - agent_2: Box(-inf, inf, (18,), float32) (shape: (18,))

Sample Observation Shapes:
  - agent_0: (18,)
  - agent_1: (18,)
  - agent_2: (18,)

Rewards received:
defaultdict(<class 'int'>, {'agent_0': -1.5823655641624477, 'agent_1': -1.5823655641624477, 'agent_2': -1.5823655641624477})
  - agent_0: -1.5823655641624477
  - agent_1: -1.5823655641624477
  - agent_2: -1.5823655641624477

Terminations:
  - agent_0: False
  - agent_1: False
  - a

### Model Architecture

MADDPG uses a simple neural network architecture for both actor and critic. The actor network consists of two hidden layers with 64 units each and ReLU activation. The output layer is a tanh layer to bound the actions between -1 and 1. We use scaling to scale the actions to the desired range, like in our case [0,1]. The critic network consists of two hidden layers with 64 units each and ReLU activation. The output layer is a single unit with linear activation.


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def init_weights(module, init_w=3e-3):
    if isinstance(module, nn.Linear):
        if hasattr(module, 'is_output') and module.is_output:
            # Use uniform initialization for the final layer
            nn.init.uniform_(module.weight, -init_w, init_w)
            nn.init.uniform_(module.bias, -init_w, init_w)
        else:  # Hidden layers
            # Use Kaiming initialization for ReLU layers
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(module.bias)

class Actor(nn.Module):
    """Actor (Policy) Model"""
    def __init__(self, state_size, action_size, hidden_sizes=(64, 64),
                 action_low=-1.0, action_high=1.0):
        """
        Initialize parameters and build model.
        
        Args:
            state_size (int): Dimension of each state
            action_size (int): Dimension of each action
            hidden_sizes (tuple): Sizes of hidden layers
            action_low (float or array): Lower bound of the action space (default: -1.0)
            action_high (float or array): Upper bound of the action space (default: 1.0)
        """
        super(Actor, self).__init__()

        self.action_low = action_low
        self.action_high = action_high
        self.scale = (action_high - action_low) / 2.0
        self.bias = (action_high + action_low) / 2.0

        self.actor_network = nn.Sequential(
            nn.Linear(state_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], action_size),
            nn.Tanh()
        )

        self.actor_network[-2].is_output = True  # The Linear layer before Tanh

        self.apply(init_weights)
    
    def forward(self, state):
        """Build an actor (policy) network that maps states -> actions"""
        x = self.actor_network(state)  # Output is in range [-1, 1]
        
        # Scale from [-1, 1] to [action_low, action_high]
        return self._scale_action(x)
    
    def _scale_action(self, action):
        """Scale action from [-1, 1] to [action_low, action_high]"""
        return self.scale * action  + self.bias
    

class Critic(nn.Module):
    """Critic (Value) Model"""
    def __init__(self, state_size, action_size, hidden_sizes=(64, 64)):
        """
        Initialize parameters and build model.
        
        Args:
            state_size (int): Dimension of each state
            action_size (int): Dimension of each action
            hidden_sizes (tuple): Sizes of hidden layers
        """
        super(Critic, self).__init__()
        
        # Build the network - concatenate state and action at the first layer
        self.critic_network = nn.Sequential(
            nn.Linear(state_size + action_size, hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], 1)
        )

        self.apply(init_weights)

    def forward(self, state, action):
        """Build a critic (value) network that maps (state, action) pairs -> Q-values"""
        x = torch.cat((state, action), dim=1)
        return self.critic_network(x) # Q-value


### Replay Buffer - (multi-agent)

The replay buffer storage is used to store the experiences of the agents. The buffer is used to sample a batch of experiences and train the agents. Following components are used in the replay buffer per experience tuple:

* observations -  $\vec{o} = {o_1}, \dots, {o_N} $
* actions - $\vec{a} = {a_1}, \dots, {a_N} $
* rewards - $\vec{r} = {r_1}, \dots, {r_N} $
* next observations - $\vec{o}’ = {o_1}’, \dots, {o_N}’ $
* dones(terminations) - $\vec{d} = {d_1}, \dots, {d_N} $

At each step $t$, the agents interact with the environment and store the experiences in the replay buffer:

$$ (\vec{o}_t, \vec{a}_t, \vec{r}_t, \vec{o}’_t, \vec{d}_t) $$

In [25]:
import numpy as np
import torch
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ReplayBuffer:
    """Replay Buffer for multi agent environments with per-agent storage in separate arrays."""

    def __init__(self, buffer_size, batch_size, agents, state_sizes, action_sizes):
        """
        Initialize the ReplayBuffer with per-agent storage.
        
        Args:
            buffer_size (int): Maximum size of the buffer
            batch_size (int): Size of each training batch
            agents (list): List of agent IDs (e.g., from env.agents)
            state_sizes (list): List of state sizes per agent (e.g., [14, 10, 10])
            action_sizes (list): List of action sizes per agent (e.g., [7, 4, 5])
        """
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.agents = agents
        self.num_agents = len(agents)
        self.state_sizes = state_sizes
        self.action_sizes = action_sizes

        # Initialize per-agent buffers as separate arrays
        self.states_buffer = []
        self.actions_buffer = []
        self.rewards_buffer = []
        self.next_states_buffer = []
        self.dones_buffer = []

        # Create separate buffers for each agent
        for i in range(self.num_agents):
            self.states_buffer.append(np.zeros((buffer_size, state_sizes[i]), dtype=np.float32))
            self.actions_buffer.append(np.zeros((buffer_size, action_sizes[i]), dtype=np.float32))
            self.rewards_buffer.append(np.zeros(buffer_size, dtype=np.float32))
            self.next_states_buffer.append(np.zeros((buffer_size, state_sizes[i]), dtype=np.float32))
            self.dones_buffer.append(np.zeros(buffer_size, dtype=np.uint8))
        
        self.position = 0
        self.size = 0

    def add(self, states, actions, rewards, next_states, dones):
        """
        Add a new experience to the buffer.
        
        Args:
            states (list): List of states per agent (variable sizes)
            actions (list): List of actions per agent (variable sizes)
            rewards (list or np.ndarray): Rewards per agent [num_agents]
            next_states (list): List of next states per agent (variable sizes)
            dones (list or np.ndarray): Done flags per agent [num_agents]
        """
        # Store experience for each agent
        for i in range(self.num_agents):
            self.states_buffer[i][self.position] = states[i]
            self.actions_buffer[i][self.position] = actions[i]
            # Store reward and done as scalars
            self.rewards_buffer[i][self.position] = rewards[i]
            self.next_states_buffer[i][self.position] = next_states[i]
            self.dones_buffer[i][self.position] = dones[i]
        
        # Update position and size
        self.position = (self.position + 1) % self.buffer_size
        self.size = min(self.size + 1, self.buffer_size)
    
    def sample(self):
        """
        Sample a batch of experiences from the buffer.
        
        Returns:
            tuple: (states_batch, actions_batch, rewards_batch, next_states_batch, dones_batch, 
                   states_full, next_states_full, actions_full)
                  Each is a tensor with appropriate shape for MADDPG training
        """
        # Sample indices
        indices = np.random.choice(self.size, self.batch_size, replace=False)
        
        # Initialize tensors for each agent
        states_batch = []
        actions_batch = []
        rewards_batch = []
        next_states_batch = []
        dones_batch = []
        
        # Collect experiences for each agent
        for i in range(self.num_agents):
            # Get data for this agent
            agent_states = self.states_buffer[i][indices]  # [batch_size, state_size_i]  
            agent_actions = self.actions_buffer[i][indices]  # [batch_size, action_size_i]
            agent_rewards = self.rewards_buffer[i][indices]  # [batch_size]
            agent_next_states = self.next_states_buffer[i][indices]  # [batch_size, state_size_i]
            agent_dones = self.dones_buffer[i][indices]  # [batch_size]
            
            # Convert to tensors
            states_batch.append(torch.tensor(agent_states).float().to(device))  # [agent_idx, batch_size, state_size_i]
            actions_batch.append(torch.tensor(agent_actions).float().to(device))  # [agent_idx, batch_size, action_size_i]
            rewards_batch.append(torch.tensor(agent_rewards).float().to(device))  # [agent_idx, batch_size]
            next_states_batch.append(torch.tensor(agent_next_states).float().to(device))  # [agent_idx, batch_size, state_size_i]
            dones_batch.append(torch.tensor(agent_dones).float().to(device))  # [agent_idx, batch_size]
        
        # Stack rewards and dones directly without squeeze
        rewards_batch = torch.stack(rewards_batch).unsqueeze(-1)  # [num_agents, batch_size, 1]
        dones_batch = torch.stack(dones_batch).unsqueeze(-1)  # [num_agents, batch_size, 1]
        
        # Create full state and action tensors for centralized critic
        states_full = torch.cat(states_batch, dim=-1)  # [batch_size, sum(state_sizes)]
        next_states_full = torch.cat(next_states_batch, dim=-1)  # [batch_size, sum(state_sizes)]
        actions_full = torch.cat(actions_batch, dim=-1)  # [batch_size, sum(action_sizes)]
        
        return (states_batch, actions_batch, rewards_batch, next_states_batch, 
                dones_batch, states_full, next_states_full, actions_full)

    def __len__(self):
        """Return the current size of the buffer."""
        return self.size
        


### DDPG Agent - (multi-agent)

Define individual DDPG agents for each agent in the environment. Each agent has its own actor and centralized critic.


In [26]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DDPGAgent:
    """
    DDPG Agent with Actor and Centralized Critic Networks.
    """
    def __init__(self, state_size, action_size, total_state_size, total_action_size, hidden_sizes=(64, 64), 
                 actor_lr=1e-4, critic_lr=1e-3, tau=1e-3,
                 action_low=None, action_high=None):
        """
        Initialize a DDPG agent.
        
        Args:
            state_size (int): Dimension of the state space
            action_size (int): Dimension of the action space
            total_state_size (int): Total dimension of all agents' states (for centralized critic)
            total_action_size (int): Total dimension of all agents' actions (for centralized critic)
            hidden_sizes (tuple): Sizes of hidden layers for networks
            actor_lr (float): Learning rate for the actor
            critic_lr (float): Learning rate for the critic (not used here, kept for compatibility)
            tau (float): Soft update parameter
            action_low (float or array): Lower bound of the action space (default: -1.0)
            action_high (float or array): Upper bound of the action space (default: 1.0)
        """
        self.state_size = state_size
        self.action_size = action_size
        self.tau = tau

        # Set action bounds
        self.action_low = -1.0 if action_low is None else action_low
        self.action_high = 1.0 if action_high is None else action_high
        self.action_range = self.action_high - self.action_low
        
        # Actor Networks (Local and Target)
        self.actor = Actor(state_size, action_size, hidden_sizes, 
                          action_low=self.action_low, 
                          action_high=self.action_high).to(device)
        self.actor_target = Actor(state_size, action_size, hidden_sizes,
                                 action_low=self.action_low, 
                                 action_high=self.action_high).to(device)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)

        # Critic Networks (Local and Target) - Centralized Critic
        self.critic = Critic(total_state_size, total_action_size, hidden_sizes).to(device)
        self.critic_target = Critic(total_state_size, total_action_size, hidden_sizes).to(device)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)
        
        self.hard_update(self.critic_target, self.critic)
        self.hard_update(self.actor_target, self.actor)
      
    def act(self, state, add_noise=True, noise_scale=1.0):
        """
        Returns actions for given state as per current policy.
        
        Args:
            state: Current state
            add_noise (bool): Whether to add noise for exploration
            noise_scale (float): Gaussian noise scale
        """
        state = torch.from_numpy(state).float().to(device)
        
        self.actor.eval()
        with torch.no_grad():
            # Get action from network (already scaled to [action_low, action_high])
            action = self.actor(state).cpu().data.numpy()
        self.actor.train()
        
        if add_noise:
            # Scale noise by action range and noise_scale
            scaled_noise = np.random.normal(0, noise_scale * self.action_range, size=action.shape)
            action += scaled_noise
            
        # Clip to [action_low, action_high] range
        return np.clip(action, self.action_low, self.action_high)

    def act_target(self, state):
        """
        Returns actions for given state as per current target policy.
        Keeps gradients for learning.
        Args:
            state: Current state (tensor)
        Returns:
            action: Action from target policy (tensor)
        """
        # Assume state is already a tensor
        if not isinstance(state, torch.Tensor):
            state = torch.from_numpy(state).float().to(device)
            
        # Return tensor directly (with gradients)
        return self.actor_target(state)
    
    def hard_update(self, target, source):
        """Hard update model parameters.
        θ_target = θ_source
        """
        for target_param, param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(param.data)
    

### MADDPG Class - Orchestrating the training

The MADDPG class is the main class that orchestrates the training of the agents.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import os

from pettingzoo import mpe

from helpers.utils import Logger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MADDPG:
    """
    Multi-Agent Deep Deterministic Policy Gradient (MADDPG) implementation
    """
    def __init__(self, 
                env_name: str = "simple_spread_v3",
                buffer_size=int(1e6),
                batch_size=512,
                hidden_sizes=(64, 64),
                actor_lr=1e-4, critic_lr=1e-3, gamma=0.99, tau=1e-3,
                noise_scale=0.3, min_noise=0.05, noise_decay_steps=int(3e5),
                use_noise_decay=True,
                warmup_steps=2000,
                update_every=25,
                max_steps=25,
                device: torch.device = torch.device("cpu"), use_logger = True):
        """
        Initialize a MADDPG agent.
        
        Args:
            env_name (str): PettingZoo MPE environment name
            buffer_size (int): Maximum size of the replay buffer
            batch_size (int): Size of each training batch
            hidden_sizes (tuple): Sizes of hidden layers for networks
            actor_lr (float): Learning rate for the actor
            critic_lr (float): Learning rate for the critic
            gamma (float): Discount factor
            tau (float): Soft update parameter
            noise_scale (float): Initial noise scale
            min_noise (float): Minimum noise scale
            noise_decay_steps (int): Number of steps to decay noise to min_noise
            warmup_steps (int): Number of warmup steps before learning starts
            use_noise_decay (bool): Whether to decay noise
            update_every (int): Update networks every n steps
            max_steps (int): Maximum number of steps per episode
            device (torch.device): Device to run the networks on (default: cpu)
        """
        try:
            env_func = getattr(mpe, env_name)  # Get the environment function (e.g., simple_spread_v3)
            self.env = env_func.parallel_env(max_cycles=max_steps, continuous_actions=True)
            self.eval_env = env_func.parallel_env(max_cycles=max_steps, continuous_actions=True)
        except AttributeError:
            raise ValueError(f"Environment '{env_name}' not found in pettingzoo.mpe. "
                            "Check the name (e.g., 'simple_spread_v3', 'simple_tag_v3').")

        self.device = device

        # Get environment information
        self.env.reset() # to get agents information
        self.agent_ids = self.env.agents
        self.num_agents = len(self.agent_ids)
        self.state_sizes = [self.env.observation_space(agent).shape[0] for agent in self.agent_ids]
        self.action_sizes = [self.env.action_space(agent).shape[0] for agent in self.agent_ids]
        # We trust all agents have the same action space
        self.action_low = self.env.action_space(self.agent_ids[0]).low[0].item()
        self.action_high = self.env.action_space(self.agent_ids[0]).high[0].item()
        self.env.close() # close the environment

        # hyperparameters
        self.gamma = gamma
        self.tau = tau
        self.update_every = update_every
        self.noise_scale = noise_scale
        self.min_noise = min_noise
        self.noise_decay_steps = noise_decay_steps
        self.use_noise_decay = use_noise_decay
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        
        # Calculate total state and action sizes for centralized critic
        self.total_state_size = sum(self.state_sizes)
        self.total_action_size = sum(self.action_sizes)
        
        # Create agents
        self.ddpg_agents = []
        for i in range(self.num_agents):
            agent = DDPGAgent(
                self.state_sizes[i], 
                self.action_sizes[i],
                self.total_state_size,
                self.total_action_size, 
                hidden_sizes=hidden_sizes,
                actor_lr=actor_lr, 
                critic_lr=critic_lr, 
                tau=tau,
                action_low=self.action_low,
                action_high=self.action_high
            )
            self.ddpg_agents.append(agent)
        
        # Replay memory
        self.replay_memory = ReplayBuffer(buffer_size, 
                                          batch_size, 
                                          self.agent_ids, 
                                          self.state_sizes, 
                                          self.action_sizes)
        if use_logger:
            # Initialize logger
            run_name = (
                f"alr{actor_lr}_clr{critic_lr}_bs{batch_size}_"
                f"hs{hidden_sizes}_g{self.gamma}_t{tau}_ns{noise_scale}_mn{min_noise}_nds{noise_decay_steps}_"
                f"ue{update_every}_ms{max_steps}"
            )
            run_name = "".join(run_name)
            self.logger = Logger(run_name=run_name, env=self.env.metadata['name'], algo="MADDPG")
            # Log hyperparameters
            self.logger.log_hyperparameters({
                "buffer_size": buffer_size,
                "batch_size": batch_size,
                "hidden_sizes": hidden_sizes,
                "max_steps": max_steps,
                "actor_lr": actor_lr,
                "critic_lr": critic_lr,
                "gamma": self.gamma,
                "tau": self.tau,
                "noise_scale": noise_scale,
                "min_noise": min_noise,
                "noise_decay_steps": noise_decay_steps,
                "use_noise_decay": use_noise_decay,
                "warmup_steps": warmup_steps,
                "update_every": update_every
            })
            

    def act(self, states, add_noise=True, noise_scale=0.0):
        """
        Get actions from all agents based on current policy.
        Args:
            states (list): List of states for each agent
            add_noise (bool): Whether to add noise for exploration
            noise_scale (float): Gaussian noise scale
        """
        actions = [agent.act(state, add_noise, noise_scale) 
                for agent, state in zip(self.ddpg_agents, states)]
        return actions

    def _learn(self, global_step = 0):
        """
        Update policy and value parameters for all agents using a batch of experience tuples.
        """
        # Learn for each agent
        for i in range(self.num_agents):
            # Learn for each agent
            critic_loss, actor_loss = self._learn_agent(i)
    
            # Log losses to TensorBoard
            self.logger.add_scalar(f'{self.agent_ids[i]}/critic_loss', critic_loss, global_step)
            self.logger.add_scalar(f'{self.agent_ids[i]}/actor_loss', actor_loss, global_step)
        
        # Update target networks
        self._update_targets()

    def _learn_agent(self, agent_idx):
        """
        Update policy and value parameters for a specific agent using given batch of experience tuples.

        Args:
            agent_idx (int): Index of the agent to update
        Returns:
            critic_loss (float): Loss of the critic network
            actor_loss (float): Loss of the actor network
        """
        # Sample a batch of experiences
        states, actions, rewards, next_states, dones, \
            states_full, next_states_full, actions_full = self.replay_memory.sample()

        current_agent = self.ddpg_agents[agent_idx]
    
        # Extract the agent's specific rewards and dones
        agent_rewards = rewards[agent_idx]
        agent_dones = dones[agent_idx]
    
        # ---------------------------- update centralized critic ---------------------------- #
        with torch.no_grad():
            # Get predicted next actions for all agents using target networks
            next_actions_list = self._act_target(next_states)
    
            # Concatenate next actions for all agents (they're already tensors)
            next_actions_full = torch.cat(next_actions_list, dim=1)

            # Compute target Q-value
            Q_targets_next = current_agent.critic_target(next_states_full, next_actions_full)
        
            # Compute Q targets for current states (y_i)
            Q_targets = agent_rewards + (self.gamma * Q_targets_next * (1 - agent_dones))

        # Compute critic loss
        Q_expected = current_agent.critic(states_full, actions_full)
    
        critic_loss = F.mse_loss(Q_expected, Q_targets)

        # Update the critic for the current agent
        current_agent.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(current_agent.critic.parameters(), 1.0)
        current_agent.critic_optimizer.step()
    
        # ---------------------------- update actor ---------------------------- #
    
        # Compute actor loss
        actions_pred = []
        for i, agent in enumerate(self.ddpg_agents):
            if i == agent_idx:
                actions_pred.append(current_agent.actor(states[i]))
            else: # Detach actions from other agents to prevent gradient flow
                actions_pred.append(actions[i].detach())

        actions_full_pred = torch.cat(actions_pred, dim=1)
    
        # Compute actor loss using the agent's critic
        actor_loss = -current_agent.critic(states_full, actions_full_pred).mean()

        # Update the actor for the current agent
        current_agent.actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(current_agent.actor.parameters(), 0.5)
        current_agent.actor_optimizer.step()
    
        return critic_loss.item(), actor_loss.item()


    def _rollout_step(self, observations, noise_scale=0.0):
        """
        Take a step in the environment using the current policy and store the experience in the replay buffer.
        Note: Called it rollout_step, because collect_rollout is widely used in RL literature.
        Args:
            observations (dict): Dictionary of observations for each agent
            noise_scale (float): Noise scale for exploration
        Returns:
            next_observations (dict): Dictionary of next observations for each agent
            rewards (dict): Dictionary of rewards for each agent
            episode_done (bool): Whether the episode is done
        """
        # Get states for all agents
        states_list = [np.array(observations[agent], dtype=np.float32) for agent in self.agent_ids]
        
        # Get actions for all agents
        actions_list = self.act(states_list, add_noise=True, noise_scale=noise_scale)
        actions = {agent: action for agent, action in zip(self.agent_ids, actions_list)}
        
        # Take a step in the environment
        next_observations, rewards, terminations, truncations, _ = self.env.step(actions)
        
        # Check if episode is done
        dones = [terminations[agent] or truncations[agent] for agent in self.agent_ids]
        episode_done = any(dones)
        
        # Prepare data to store in replay buffer (we care only terminations)
        rewards_array = np.array([rewards[agent] for agent in self.agent_ids], dtype=np.float32)
        next_states_list = [np.array(next_observations[agent], dtype=np.float32) for agent in self.agent_ids]
        terminations_array = np.array([terminations[agent] for agent in self.agent_ids], dtype=np.uint8)
        
        # Store experience in replay buffer
        self.replay_memory.add(
            states=states_list,
            actions=actions_list,
            rewards=rewards_array,
            next_states=next_states_list,
            dones=terminations_array
        )

        return next_observations, rewards, episode_done
    
    def train(self, total_timesteps:int = 1e6, eval_interval=5000):
        """
        Train the MADDPG agent in the environment.

        Args:
            total_timesteps (int): Total number of training steps
            eval_interval (int): Evaluate every n steps
        """

        agent_rewards = [[] for _ in range(self.num_agents)]
        episode_rewards = np.zeros(self.num_agents)
        noise_decay = (self.noise_scale - self.min_noise) / self.noise_decay_steps
        noise_scale = self.noise_scale
        best_score = -np.inf

        model_path = os.path.join(self.logger.dir_name, 'torch.model')
        best_model_path = os.path.join(self.logger.dir_name, "best-torch.model")

        # Reset environment and agents
        observations, _ = self.env.reset()

        for  global_step in range(1, total_timesteps + 1):

            # collect experience
            next_observations, rewards, episode_done = self._rollout_step(observations, noise_scale)

             # Update observations and rewards
            episode_rewards += np.array(list(rewards.values())) 
            
            # Handle episode end
            if episode_done or (global_step % self.max_steps == 0):  # Reset after max_steps if not done
                for i, reward in enumerate(episode_rewards):
                    agent_rewards[i].append(reward)
                    self.logger.add_scalar(f"{self.agent_ids[i]}/episode_reward", reward, global_step)
                self.logger.add_scalar('train/total_reward', np.sum(episode_rewards), global_step)
                self.logger.add_scalar(f"noise/scale", noise_scale, global_step)
                observations, _ = self.env.reset()
                episode_rewards = np.zeros(self.num_agents)
            else:
                observations = next_observations
                
            # Warm up the agent before learning
            if global_step > self.warmup_steps: 
                # Learn every update_every steps
                if global_step % self.update_every == 0:
                    self._learn(global_step)
        
                # Update noise scale based on iteration number
                if self.use_noise_decay:
                    noise_scale = max(
                        self.min_noise,
                        noise_scale - noise_decay
                    )
        
            # Evaluate and save
            if global_step % eval_interval == 0 or global_step == total_timesteps:
                self.save(model_path)
                avg_eval_rewards = self._evaluate(num_episodes=10, global_step=global_step)
                np.save(os.path.join(self.logger.dir_name, "agent_rewards.npy"), agent_rewards)
                score = np.sum(avg_eval_rewards)
                if score > best_score:
                    best_score = score
                    self.save(best_model_path)
        

    def _act_target(self, states):
        """
        Get actions from all agents based on target policies.
        Args:
            states: States for all agents [batch_size, num_agents, state_size]
        Returns:
            actions: List of actions for each agent"
        """
        actions = [agent.act_target(state) for agent, state in zip(self.ddpg_agents, states)]
    
        return actions

    def _soft_update(self, target, source):
        """
        Soft update model parameters.
        θ_target = τ*θ_source + (1 - τ)*θ_target 
        Args:
            target: Model with weights to update
            source: Model with weights to copy from
        """
        for target_param, source_param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(self.tau * source_param.data + (1.0 - self.tau) * target_param.data)
    
    def _update_targets(self):
        """
        Soft update target networks for all agents.
        This should be called after all agents have been updated.
        """
        # print("\nUpdating target networks:")
        for i, agent in enumerate(self.ddpg_agents):
            # Perform soft update
            self._soft_update(agent.actor_target, agent.actor)
            self._soft_update(agent.critic_target, agent.critic)
    
    def _evaluate(self, num_episodes=10, global_step=0):
        """Run evaluation episodes and return average rewards."""
        eval_rewards = [] 
        for _ in range(num_episodes):
            observations, _ = self.eval_env.reset()
            agents = self.eval_env.agents
            done = False 
            episode_rewards = np.zeros(len(agents))
            while not done:
                states_list = [np.array(observations[agent], dtype=np.float32) for agent in agents]
                actions_list = self.act(states_list, add_noise=False)  # No noise for eval
                actions = {agent: action for agent, action in zip(agents, actions_list)}
                next_observations, rewards, terminations, truncations, _ = self.eval_env.step(actions)
                episode_rewards += np.array(list(rewards.values()))
                dones = [terminations[agent] or truncations[agent] for agent in agents]
                done = any(dones)
                observations = next_observations
            eval_rewards.append(episode_rewards) # (num_eval_episodes, num_envs)
        
        avg_eval_rewards = np.mean(eval_rewards, axis=0) # (num_envs,)

        for i, avg_reward in enumerate(avg_eval_rewards):
            self.logger.add_scalar(f'{agents[i]}/eval_reward', avg_reward, global_step) 

        total_eval_reward = np.sum(eval_rewards) / num_episodes
        self.logger.add_scalar('eval/total_reward', total_eval_reward, global_step) 
        print(f"Step {global_step}, Eval rewards: {avg_eval_rewards}, Sum: {total_eval_reward}")     
        return avg_eval_rewards

    def save(self, path):
        """
        Save all agent models to a single file.
        
        Args:
            path (str): Path to save the models
        """
        os.makedirs(os.path.dirname(path), exist_ok=True)
        
        # Create a dictionary to store all models
        models_dict = {}
        
        for i, agent in enumerate(self.ddpg_agents):
            # Save actor and critic models
            models_dict[f'agent_{i}_actor'] = agent.actor.state_dict()
            models_dict[f'agent_{i}_critic'] = agent.critic.state_dict()
        
        # Save all models to a single file
        torch.save(models_dict, path)
        print(f"Models saved to {path}")
    
    def load(self, path):
        """
        Load all agent models from a single file.
        
        Args:
            path (str): Path to load the models from
        """
        if not os.path.exists(path):
            print(f"Warning: No model file found at {path}")
            return
            
        # Load the dictionary containing all models
        models_dict = torch.load(path, weights_only=False)
        
        for i, agent in enumerate(self.ddpg_agents):
            # Load actor model
            actor_key = f'agent_{i}_actor'
            if actor_key in models_dict:
                agent.actor.load_state_dict(models_dict[actor_key])
                agent.actor_target.load_state_dict(models_dict[actor_key])
                print(f"Loaded actor model for agent {i}")
            
            # Load critic model
            critic_key = f'agent_{i}_critic'
            if critic_key in models_dict:
                agent.critic.load_state_dict(models_dict[critic_key])
                agent.critic_target.load_state_dict(models_dict[critic_key])
                print(f"Loaded critic model for agent {i}")
        
        print(f"All models loaded from {path}") 


### Training Example - Simple Spread Environment

Simple Spread is a simple environment where agents are rewarded for taking actions that bring them closer to the landmarks. The environment is a 2D grid-world with agents and landmarks.

In [30]:

# Create an instance of the MADDPG agent
maddpg = MADDPG(env_name="simple_spread_v3",
                buffer_size=int(1e6), 
                batch_size=256, 
                hidden_sizes=(64, 64), 
                actor_lr=1e-3, 
                critic_lr=2e-3, 
                gamma=0.95, 
                tau=1e-2, 
                noise_scale=0.3, 
                min_noise=0.01, 
                noise_decay_steps=int(5e5), 
                use_noise_decay=True, 
                warmup_steps=20000, 
                update_every=15,
                max_steps=25, 
                device=torch.device("cpu"))

# Train the agent
maddpg.train(total_timesteps=1000000, eval_interval=5000)


----------------------------------------------------------------------------
| Hyperparams                    | Values                                  |
----------------------------------------------------------------------------
| buffer_size                    | 1000000                                 |
| batch_size                     | 256                                     |
| hidden_sizes                   | (64, 64)                                |
| max_steps                      | 25                                      |
| actor_lr                       | 0.001                                   |
| critic_lr                      | 0.002                                   |
| gamma                          | 0.95                                    |
| tau                            | 0.01                                    |
| noise_scale                    | 0.3                                     |
| min_noise                      | 0.01                                    |

### Evaluation of the trained model

In [35]:
import imageio
from PIL import Image, ImageDraw, ImageFont
from pettingzoo.mpe import simple_spread_v3
import numpy as np 

# Init simple_spread_v3 environment
env = simple_spread_v3.parallel_env(max_cycles=25, continuous_actions=True,  render_mode="rgb_array")
# Create maddpg agent with correct environment name and hidden sizes
maddpg = MADDPG(env_name="simple_spread_v3",
                hidden_sizes=(64, 64), 
                device=torch.device("cpu"),
                use_logger=False)

# Load the best model
maddpg.load("./runs/simple_spread_v3/MADDPG/alr0.001_clr0.002_bs256_hs(64, 64)_g0.95_t0.01_ns0.3_mn0.01_nds500000_ue15_ms25/torch.model")


def add_text_to_frame(frame, text):
    """Add simple text to a frame using PIL."""
    # Convert numpy array to PIL Image
    img = Image.fromarray(frame)
    
    # Create a drawing context
    draw = ImageDraw.Draw(img)
    
    # Try to use a standard font, fall back to default if not available
    try:
        font = ImageFont.truetype("Arial", 16)
    except IOError:
        font = ImageFont.load_default()
    
    # Add text at the top left corner
    position = (10, 10) 

    # Draw text with a black outline for better visibility
    
    draw.text(position, text, font=font, fill=(0, 0, 0))    
    
    # Convert back to numpy array
    return np.array(img)

# Create gif and evaluate the agent
def evaluate(env, maddpg, record_gif=True, num_eval_episodes=10):
    """Run evaluation episodes and return average rewards."""
    eval_rewards = [] 
    all_frames = []  if record_gif else None
    for episode in range(1, num_eval_episodes + 1):
        observations, _ = env.reset()
        agents = env.agents
        done = False 
        episode_rewards = np.zeros(len(agents))
        episode_frames = [] if record_gif else None
        while not done:
            states_list = [np.array(observations[agent], dtype=np.float32) for agent in agents]
            actions_list = maddpg.act(states_list, add_noise=False)  # No noise for eval
            actions = {agent: action for agent, action in zip(agents, actions_list)}
            next_observations, rewards, terminations, truncations, _ = env.step(actions)
            episode_rewards += np.array(list(rewards.values()))
            dones = [terminations[agent] or truncations[agent] for agent in agents]
            done = any(dones)
            if record_gif: 
                frame = env.render()
                text = f"Ep {episode} - R: {np.sum(episode_rewards):.1f}"
                labeled_frame = add_text_to_frame(frame, text)
                
                episode_frames.append(labeled_frame)
                all_frames.append(labeled_frame)
            observations = next_observations
        eval_rewards.append(episode_rewards) # (num_eval_episodes, num_envs)

        print(f"Episode {episode}, Rewards: {episode_rewards}, Total: {np.sum(episode_rewards)}")

        if record_gif and episode < num_eval_episodes and episode_frames:
             # Get the shape of the frames
            frame_shape = episode_frames[0].shape
            # Create a black frame with the same dimensions
            black_frame = np.zeros(frame_shape, dtype=np.uint8)
            # Add simple episode text
            next_text = f"Episode {episode+1}"
            next_frame = add_text_to_frame(black_frame, next_text)
            # Add separator frames
            separator_frames = int(10)  # 10 frames per second
            for _ in range(separator_frames):
                all_frames.append(next_frame)
    
    if record_gif and all_frames:
        combined_gif_path = os.path.join('.', f"maddpg_all_episodes.gif")
        try:
            imageio.mimsave(combined_gif_path, all_frames, duration=0.1)  # 100ms per frame
            print(f"Saved combined GIF with all episodes to {combined_gif_path}")
        except Exception as e:
            print(f"Error saving combined GIF: {e}")
    
    avg_eval_rewards = np.mean(eval_rewards, axis=0) # (num_envs,)
    total_eval_reward = avg_eval_rewards.sum()
    print(f"Eval rewards: {avg_eval_rewards}, Total: {total_eval_reward}")          
    
    return avg_eval_rewards


evaluate(env, maddpg, record_gif=True, num_eval_episodes=10)



Loaded actor model for agent 0
Loaded critic model for agent 0
Loaded actor model for agent 1
Loaded critic model for agent 1
Loaded actor model for agent 2
Loaded critic model for agent 2
All models loaded from ./runs/simple_spread_v3/MADDPG/alr0.001_clr0.002_bs256_hs(64, 64)_g0.95_t0.01_ns0.3_mn0.01_nds500000_ue15_ms25/torch.model
Episode 1, Rewards: [-14.26435476 -14.76435476 -14.76435476], Total: -43.79306428182662
Episode 2, Rewards: [-12.80331066 -13.30331066 -13.30331066], Total: -39.4099319756786
Episode 3, Rewards: [ -6.47209633 -10.47209633 -10.97209633], Total: -27.91628900243643
Episode 4, Rewards: [-11.05790785 -13.05790785 -13.05790785], Total: -37.17372354299903
Episode 5, Rewards: [-11.41093301 -11.41093301 -11.41093301], Total: -34.232799017240886
Episode 6, Rewards: [-11.17191622 -11.17191622 -11.17191622], Total: -33.515748669228635
Episode 7, Rewards: [-9.9353777 -9.9353777 -9.9353777], Total: -29.8061331011289
Episode 8, Rewards: [-11.27441711 -11.27441711 -11.2744

array([-10.91833677, -11.71833677, -11.61833677])

<div style="text-align:center">
    <img src="../../assets/gifs/maddpg_simple_spread.gif" width="600" height="auto">
</div>